🌐 [English](dx_app.ipynb) | [한국어](dx_app.ko.ipynb) | [日本語](dx_app.ja.ipynb) | **中文**

# DEEPX Tutorial 02 - DX-APP 使用方法

在第二篇教程中,我们将介绍 DX-APP,并学习如何在 AI 应用中运行编译后的 DXNN 模型。

## 学习目标

完成本教程后,您将能够:

- 了解 DX-APP 为 **[DEEPX Model Zoo](https://developer.deepx.ai/modelzoo/) 中的每一个模型**(497 个模型变体)都提供了可直接运行的 C++ 和 Python 示例代码,并使用这些代码运行您选择的任意模型;
- 描述示例如何按任务、模型系列和变体组织,以及变体名称如何把模型文件、可执行文件和脚本关联起来;
- 只下载教程所需的模型和视频;
- 运行分类、检测、姿态和分割示例,并在笔记本中查看保存的结果;
- 在同步、异步和 PPU 执行方式之间进行选择,并通过测量确定哪种 Python 变体适合您的模型;
- 使用 `dx_tool.sh` 查询示例,并启动演示启动器;
- 导出独立的 C++ 或 Python 示例,阅读它的模型卡,并在 DX-APP 目录树之外运行它;
- 使用多模型图把多个模型串联成一条流水线,只需编辑 JSON 文件而无需修改代码。

## 什么是 DX-APP?

**DX-APP** 是面向 DEEPX NPU 的参考应用。它的核心特点是覆盖面:**[DEEPX Model Zoo](https://developer.deepx.ai/modelzoo/) 中发布的每一个模型都有与之对应、可直接运行的示例**,同时提供 C++ 和 Python 两个版本,并按任务组织(目标检测、分类、人脸检测、姿态、分割、深度等)。您在 Model Zoo 页面选好一个模型,DX-APP 中就已经包含了运行它的代码。DX-APP 帮助开发者快速搭建运行时环境,
并可作为构建和定制自己的 AI 应用的模板。

本教程使用 **DX-APP v3.3.0**,即 DX-All Suite v2.5.0 附带的版本。

更多细节请参阅 [dx_app git 仓库](https://github.com/DEEPX-AI/dx_app/blob/main/README.md)中的概述,或从[这里](https://developer.deepx.ai/download/?id=744)下载 DX-APP 用户指南!

> **注意:** 下载用户指南前,必须先登录 https://developer.deepx.ai/。

让我们看看 DX-APP 的文件结构:

In [ ]:
# Load the shared DEEPX tutorial paths (identical first code cell in every notebook).
from pathlib import Path; import importlib, sys
sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "tutorial_paths.py").is_file())))
import tutorial_paths; importlib.reload(tutorial_paths)  # pick up updates without restarting the kernel
from tutorial_paths import setup_tutorial
import shlex

paths = setup_tutorial(needs=["dx_rt", "dx_app", "npu"])
paths.require(*["dx_rt", "dx_app", "npu"])   # stop here with the fix if a prerequisite is missing
%cd {DX_ALL_SUITE_DIR}


In [ ]:
!tree -L 1 dx-runtime

In [ ]:
!tree -L 1 dx-runtime/dx_app
!tree -L 1 dx-runtime/dx_app/bin
#!tree -L 2 dx-runtime/dx_app/src

**DX-APP** 演示程序经过优化,可用最少的配置在 DEEPX NPU 上展示预编译模型。
每个演示程序代表一种常见的 AI 任务,可以使用图像、视频或实时摄像头
输入来执行。

<img src="assets/ai-model-type.png" style="max-width: 1000px;">

---
**DX-APP 仓库布局:**
```shell
dx_app/
├── src/
│   ├── cpp_example/                    # C++ end-to-end examples (497 model variants across 29 tasks)
│   │   ├── <task>/<family>/<variant>/  #   One folder per .dxnn file: entry points, factory, config.json, README.md
│   │   ├── multi_model_graph/          #   Multi-model graph CLI and sample graph JSON files (section 4)
│   │   └── common/                     # Shared C++ runtime layer
│   │       ├── base/                   #   Abstract interfaces (factories, pre/post-processors, visualizers, inputs)
│   │       ├── processors/             #   Shared processors
│   │       ├── runner/                 #   Task-specific sync/async runner pairs
│   │       ├── inputs/                 #   Image, video, camera, and RTSP input sources
│   │       ├── visualizers/            #   Task-specific visualizers
│   │       ├── config/                 #   ModelConfig loader
│   │       ├── graph/                  #   Multi-model graph engine
│   │       ├── registry/               #   Model registry used by the graph engine
│   │       ├── trackers/               #   IoU tracker
│   │       ├── third_party/            #   Header-only third-party libraries
│   │       └── utility/                #   Labels, preprocessing, profiling, and run utilities
│   ├── python_example/                 # Python end-to-end examples (the same 497 variants and 29 tasks)
│   │   ├── <task>/<family>/<variant>/  #   Four scripts per variant, config.json, and the model card files
│   │   └── common/                     # Shared Python runtime layer (same structure as the C++ one)
│   ├── postprocess/                    # C++ post-processing (consumed by pybind11 bindings)
│   ├── utility/                        # Shared support code used by build flow
│   └── bindings/
│       └── python/
│           ├── dx_postprocess/         # pybind11 bindings wrapping src/postprocess/
│           └── dx_graph/               # pybind11 module running multi-model graphs (./build.sh --all)
├── config/
│   ├── model_registry.json             # Model registry: one row per variant, the single source of truth
│   ├── test_models.conf                # Test model configuration
│   └── README.md                       # Config directory documentation
├── scripts/                            # dx_tool.sh, extract_model_package.sh, download_models.py, and helpers
├── tests/                              # pytest-based test suites
├── assets/                             # Links to downloaded models and videos
├── sample/                             # Bundled images and task-specific sample data
├── bin/                                # C++ executables generated by the build: <variant>_sync, <variant>_async
├── build.sh                            # Build script (default: the 26 demo models; --all: every variant)
├── setup.sh                            # Model, video, and dependency setup
├── run_demo.sh                         # Demo launcher: 26 demos, up to 6 execution modes
├── run_multi_model_demo.sh             # Multi-model graph scenarios and graph checks
├── install.sh                          # Dependency and OpenCV installer
└── docs/                               # Detailed documentation
```
---
**任务、模型系列和变体**

每个示例都位于 `<task>/<family>/<variant>/` 中。**变体**(variant)是 `.dxnn` 文件名去掉扩展名后的部分,属于该模型的所有内容都以它命名:

| 项目 | 以 `yolov7_640x640.dxnn` 为例 |
|---|---|
| 示例文件夹 | `src/cpp_example/object_detection/yolov7/yolov7_640x640/`,以及 `src/python_example/` 下的相同路径 |
| C++ 可执行文件 | `bin/yolov7_640x640_sync`、`bin/yolov7_640x640_async` |
| Python 脚本 | `yolov7_640x640_sync.py`、`yolov7_640x640_async.py`、`yolov7_640x640_sync_cpp_postprocess.py`、`yolov7_640x640_async_cpp_postprocess.py` |
| 下载名称 | `./setup.sh --models yolov7_640x640` |

> **从 DX-APP v3.2.x 升级而来?** 可执行文件现在以变体命名(`bin/yolov7_async` 现在是 `bin/yolov7_640x640_async`),PPU 模型移到了 `object_detection/yolo_ppu/`,并且有几个任务被重命名或拆分:`classification` 现在是 `image_classification`,`obb_detection` 改为 `oriented_object_detection`,`image_enhancement` 改为 `low_light_enhancement`,`face_alignment` 改为 `face_landmark`,而 `embedding` 拆分为 `face_recognition`、`image_retrieval`、`visual_place_recognition` 和 `zero_shot_image_classification`。

**29 个任务目录**(`cpp_example` 和 `python_example` 中相同):

- 检测:`object_detection`、`oriented_object_detection`、`face_detection`、`hand_detection`、`mainbody_detection`、`3d_object_detection`、`anomaly_detection`
- 分割:`instance_segmentation`、`semantic_segmentation`、`zero_shot_instance_segmentation`、`panoptic_driving_perception`、`image_matting`
- 姿态与关键点:`pose_estimation`、`hand_landmark`、`face_landmark`、`keypoint_detection`、`object_pose_estimation`
- 分类与属性:`image_classification`、`zero_shot_image_classification`、`person_attribute`、`face_attribute`
- 识别与检索:`face_recognition`、`person_reid`、`image_retrieval`、`visual_place_recognition`
- 图像复原与深度:`image_denoising`、`super_resolution`、`low_light_enhancement`、`depth_estimation`

DX-APP v3.3.0 新注册了 147 个 Model Zoo 模型,包括 RT-DETR(v1、v2 和 v3)、YOLOv12、YOLOv13、预优化的 YOLO26,以及用于新任务的模型:异常检测(PatchCore、EfficientAD)、图像抠图(PP-Matting)、CLIP 图像检索、行人重识别,以及视觉地点识别(EigenPlaces)。

## 前提条件

- 已完成教程 01:使用 `--all` 安装了 DX-Runtime(DX-RT、DX-APP 及其 `venv-dx-runtime` Python 环境),NPU 以 `/dev/dxrt0` 的形式可见。
- **DX-RT 3.5.0 或更高版本。** DX-APP v3.3.0 从 Model Zoo 版本 `2_5_0` 下载模型,其 `.dxnn` 文件使用 v9 容器格式。较旧的运行时会拒绝加载这些文件(参见故障排除)。`dxrt-cli --version` 会打印已安装的版本。
- DX-APP 示例模型和视频。教程 01 的 3.3 节会下载它们;下面的下载单元格只获取仍然缺失的内容(本教程约需 300 MB 的模型,在全新主机上还需 1.2 GB 的共享示例视频压缩包)。
- 图形桌面会话为可选项。单元格以无显示方式运行示例(`--no-display --save`),并在笔记本中显示保存的图像;会打开窗口的单元格由 `SHOW_WINDOW = False` 这样的标志关闭。
- 不需要 `sudo`。预计约 45 分钟。有两个单元格各需一到两分钟:3.3 节的 C++ 构建,以及在缺少图可执行文件时 4.1 节的构建。

下一个单元格会定位 SDK、检查这些要求并打印状态表。任何标记为 `MISSING` 的项目都会附带提供它的步骤。

## 1. 准备模型和视频

1. 进入 `dx_app` 目录:

In [ ]:
# Move to the configured DX-APP directory.
%cd {DX_APP_DIR}

2. 使用 `setup.sh` 下载所需的模型和示例视频。该脚本是交互式的(它会询问要下载哪些模型类别),因此需要在终端中运行,而不是在单元格中。打开 **File > New > Terminal** 并运行:

```bash
cd <DX_ALL_SUITE_DIR>/dx-runtime/dx_app
./setup.sh --no-force
```

3. 下一个单元格会用您的实际安装路径打印同样的两条命令。

In [ ]:
# Interactive script: run these two commands in a terminal (File > New > Terminal).
print(f"cd {shlex.quote(str(DX_APP_DIR))}")
print("./setup.sh --no-force")


> ![](assets/sc-setup.png)

4. 如果您有足够的可用存储空间容纳整个 Model Zoo(数十 GB),请按 **Enter** 下载所有 Model Zoo 模型。截图来自较早的版本;当前菜单列出了更多模型和新的任务名称。

5. 否则,请使用下面的代码单元格只下载本教程需要的模型。请传入**变体名称**,即 `.dxnn` 文件名去掉扩展名的部分,例如 `yolov7_640x640`;旧的 Model Zoo 名称(例如 `YoloV7`)仍然可以使用。教程 01 的 3.3 节已经下载了这些模型;如果您已完成该节,`--no-force` 会让单元格跳过所有文件并在几秒内完成。第 3 节和第 4 节会以同样的方式获取它们额外需要的模型。

In [ ]:
# The models used in section 2, by variant name (the .dxnn file stem).
# --no-force skips files that already exist, so rerunning this cell is cheap.
SELECTED_MODELS = shlex.join([
    "resnet50_224x224",
    "yolov7_640x640",
    "yolov7_640x640_ppu",
    "yolov8-s-pose_640x640",
    "yolov8-n-seg_640x640",
])

# Preview what would be downloaded, without downloading:
#!cd "{DX_APP_DIR}" && bash setup.sh --models {SELECTED_MODELS} --dry-run

# Download the selected models only when they are missing
!cd "{DX_APP_DIR}" && bash setup.sh --models {SELECTED_MODELS} --no-force

6. 验证模型和视频都已按预期下载:

In [ ]:
# Downloaded models and videos are stored under `assets` path as a symbolic link
!cd "{DX_APP_DIR}" && tree assets

In [ ]:
# AI models converted to DXNN format

# Number of downloaded models
!cd "{DX_APP_DIR}" && ls assets/models | wc -l

# Downloaded model list
!cd "{DX_APP_DIR}" && tree assets/models

In [ ]:
# video files for demo inputs
!cd "{DX_APP_DIR}" && tree assets/videos

## 2. 运行演示程序

**DX-APP** 是一组现成的演示应用,展示如何在 DEEPX NPU 上运行编译后的模型,包括分类、检测、分割、姿态估计等多种任务。

用户可以通过探索各种 DX-APP 演示程序来学习并构建应用。

`bin/` 中的 C++ 可执行文件以变体命名:`bin/<variant>_sync` 和 `bin/<variant>_async`。它们都接受相同的输入选项(`-i` 图像、`-v` 视频、`-c` 摄像头、`-r` RTSP 流)和相同的输出选项:

| 选项 | 作用 |
|---|---|
| `--no-display` | 不打开窗口,只打印性能摘要 |
| `-s`, `--save` | 保存渲染后的结果 |
| `--save-dir <dir>` | 保存位置;每次运行都会写入一个新的子目录 |
| `-l`, `--loop <n>` | 将输入重复 `n` 次 |

本节的单元格使用 `--no-display --save` 以无显示方式运行,并在单元格下方显示保存的图像,因此它们也能通过 SSH 使用,并且永远不会阻塞笔记本。被注释掉的行会在视频或摄像头上运行同一个示例;它们会打开窗口,因此请在桌面会话中使用。

下一个单元格会创建输出目录,以及一个显示最新保存图像的小辅助函数。

In [ ]:
import time
from IPython.display import Image, Markdown, display

DEMO_OUTPUT_DIR = WORKSPACE_DIR / "dx-app-outputs"   # results saved with --save in section 2
DEMO_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Remember when each cell starts, so the helper shows only images saved by the current cell.
CELL_START = {"time": time.time()}
def _mark_cell_start(info):
    CELL_START["time"] = time.time()
ip = get_ipython()
for callback in list(ip.events.callbacks["pre_run_cell"]):
    if getattr(callback, "__name__", "") == "_mark_cell_start":
        ip.events.unregister("pre_run_cell", callback)
ip.events.register("pre_run_cell", _mark_cell_start)


def show_latest_image(folder, width=640):
    # Show the newest .jpg/.png that the current cell saved under `folder`
    # (every run saves into its own sub-directory).
    images = [p for p in Path(folder).rglob("*")
              if p.suffix.lower() in (".jpg", ".jpeg", ".png") and p.stat().st_mtime >= CELL_START["time"]]
    if not images:
        print(f"This cell saved no image under {folder}; check the command output above.")
        return
    latest = max(images, key=lambda p: p.stat().st_mtime)
    print("Saved result:", latest.relative_to(folder))
    display(Image(filename=str(latest), width=width))

### 2.1 分类
- **bin/resnet50_224x224_sync** 运行一个预训练的 ImageNet 分类模型,并在图像上绘制预测概率最高的类别。

对每个 C++ 示例来说,`-m` 都是可选的:不指定时,示例会加载其默认模型 `assets/models/<variant>.dxnn`。这些单元格仍然会传入它,以便您看清模型来自哪里。

In [ ]:
%cd {DX_APP_DIR}

# Help: the options shared by every C++ example
!./bin/resnet50_224x224_sync -h

# Image input, headless: classify one ImageNet sample and save the annotated image
!./bin/resnet50_224x224_sync -m assets/models/resnet50_224x224.dxnn -i sample/ILSVRC2012/1.jpeg --no-display --save --save-dir "{DEMO_OUTPUT_DIR}"
show_latest_image(DEMO_OUTPUT_DIR)

### 2.2 目标检测
- **bin/yolov7_640x640_sync** 和 **bin/yolov7_640x640_async** 在图像、视频或摄像头上运行 YOLOv7 目标检测器。下一个单元格会列出默认构建生成的其他 YOLO 可执行文件。

In [ ]:
%cd {DX_APP_DIR}

# Find other YOLO-based reference executables
!ls ./bin/yolo*

#### 执行模式

DX-APP 提供不同的执行模式,以满足不同的应用需求:

| 模式 | 主要特点 | 推荐用途 |
|---|---|---|
| 同步 | 逐帧顺序处理 | 简单流程和延迟检查 |
| 异步 | 流水线各阶段重叠执行 | 更高的吞吐量 |
| PPU | 卸载受支持的后处理工作 | 更低的主机 CPU 开销 |

- **同步应用**

<img src="https://github.com/DEEPX-AI/dx_rt/raw/main/docs/source/resources/09_01_Sync_Inference_Operation.png" style="max-width: 300px;">

In [ ]:
%cd {DX_APP_DIR}

# Image input, headless: the annotated image is saved and shown below
!./bin/yolov7_640x640_sync -m assets/models/yolov7_640x640.dxnn -i sample/img/sample_dog.jpg --no-display --save --save-dir "{DEMO_OUTPUT_DIR}"
show_latest_image(DEMO_OUTPUT_DIR)

# Video input, headless: 588 frames, prints the performance summary (about 20 seconds)
!./bin/yolov7_640x640_sync -m assets/models/yolov7_640x640.dxnn -v assets/videos/boat.mp4 --no-display

# On a desktop session (opens a window):
#!./bin/yolov7_640x640_sync -m assets/models/yolov7_640x640.dxnn -v assets/videos/boat.mp4
#!./bin/yolov7_640x640_sync -m assets/models/yolov7_640x640.dxnn -c 0

- **异步应用**


<img src="https://github.com/DEEPX-AI/dx_rt/raw/main/docs/source/resources/09_01_Async_Inference_Operation.png" style="max-width: 1000px;">

下一个单元格用异步可执行文件处理同一段视频。将它的 `Overall FPS` 行与上面的同步运行进行比较。

In [ ]:
%cd {DX_APP_DIR}

# Video input, headless: compare "Overall FPS" with the synchronous run above
!./bin/yolov7_640x640_async -m assets/models/yolov7_640x640.dxnn -v assets/videos/boat.mp4 --no-display

# On a desktop session (opens a window):
#!./bin/yolov7_640x640_async -m assets/models/yolov7_640x640.dxnn -v assets/videos/boat.mp4
#!./bin/yolov7_640x640_async -m assets/models/yolov7_640x640.dxnn -c 0

# RTSP input (this public stream is reachable only from Korea)
#!./bin/yolov7_640x640_async -m assets/models/yolov7_640x640.dxnn -r "rtsp://210.99.70.120:1935/live/cctv002.stream"

- **PPU 与非 PPU 对比**

下面两个单元格分别用非 PPU 模型(`bin/yolov7_640x640_async`)和 PPU 模型(`bin/yolov7_640x640_ppu_async`)处理同一段 7 秒的视频片段。当 `SHOW_WINDOW = False`(默认值)时,它们以无显示方式运行(带 `--no-display`),只打印 FPS 摘要,因此笔记本不会阻塞;在桌面会话中设置 `SHOW_WINDOW = True`,即可在 DX-APP 窗口中观看检测结果和 FPS 计数器。比较两行 FPS 输出:PPU 模型把候选框过滤和类别选择移到了 NPU 上。所有 PPU 示例都列在 `object_detection/yolo_ppu/` 下。

In [ ]:
SHOW_WINDOW = False   # True opens the DX-APP window (desktop session only); False prints the FPS summary and returns
%cd {DX_APP_DIR}

# PPU disabled
!./bin/yolov7_640x640_async -m assets/models/yolov7_640x640.dxnn -v assets/videos/drone-city-road2.mov {"" if SHOW_WINDOW else "--no-display"}

In [ ]:
SHOW_WINDOW = False   # keep the same value as in the previous cell
%cd {DX_APP_DIR}

# PPU enabled
!./bin/yolov7_640x640_ppu_async -m assets/models/yolov7_640x640_ppu.dxnn -v assets/videos/drone-city-road2.mov {"" if SHOW_WINDOW else "--no-display"}

### 2.3 姿态估计
- **bin/yolov8-s-pose_640x640_sync** 运行一个姿态估计模型,在图像、视频或摄像头中检测人并估计其身体关键点。

In [ ]:
%cd {DX_APP_DIR}

# Image input, headless
!./bin/yolov8-s-pose_640x640_sync -m assets/models/yolov8-s-pose_640x640.dxnn -i sample/img/sample_person_b.jpg --no-display --save --save-dir "{DEMO_OUTPUT_DIR}"
show_latest_image(DEMO_OUTPUT_DIR)

# On a desktop session (opens a window):
#!./bin/yolov8-s-pose_640x640_sync -m assets/models/yolov8-s-pose_640x640.dxnn -v assets/videos/dance-solo.mov
#!./bin/yolov8-s-pose_640x640_sync -m assets/models/yolov8-s-pose_640x640.dxnn -c 0

### 2.4 分割
- **bin/yolov8-n-seg_640x640_async** 运行一个实例分割模型:为它在图像、视频或摄像头中找到的每个目标生成一个边界框和一个像素掩码。

In [ ]:
%cd {DX_APP_DIR}

# Image input, headless
!./bin/yolov8-n-seg_640x640_async -m assets/models/yolov8-n-seg_640x640.dxnn -i sample/img/sample_person_b.jpg --no-display --save --save-dir "{DEMO_OUTPUT_DIR}"
show_latest_image(DEMO_OUTPUT_DIR)

# On a desktop session (opens a window):
#!./bin/yolov8-n-seg_640x640_async -m assets/models/yolov8-n-seg_640x640.dxnn -v assets/videos/blackbox-city-road.mp4
#!./bin/yolov8-n-seg_640x640_async -m assets/models/yolov8-n-seg_640x640.dxnn -c 0

### 2.5 演示启动器:使用 run_demo.sh 运行演示程序

`run_demo.sh` 可启动 **26 个演示程序**(每个任务一个),最多支持 **6 种执行模式**:C++ 同步和异步,以及四种 Python 变体。它会询问三个问题(演示程序、模式、输入),因此需要在终端中启动并回答菜单(**File > New > Terminal**):

```bash
cd <DX_ALL_SUITE_DIR>/dx-runtime/dx_app
./run_demo.sh
```

下一个单元格会用您的实际路径打印同样的命令。在显示第一个菜单之前,启动器会下载 26 个演示模型中缺失的那些,因此首次启动可能需要几分钟。它还会自行找到 `venv-dx-runtime` 的 Python,因此 Python 模式无需激活虚拟环境。

In [ ]:
# Interactive script: run these two commands in a terminal (File > New > Terminal).
print(f"cd {shlex.quote(str(DX_APP_DIR))}")
print("./run_demo.sh")


第一个菜单列出了各个演示程序(菜单本身以英文显示):

| # | 演示程序 | # | 演示程序 |
|---|---|---|---|
| 0 | 目标检测 (YOLOv7) | 13 | 图像增强 (Zero-DCE) |
| 1 | 目标检测 (YOLOv11N) | 14 | 嵌入 (ArcFace) |
| 2 | 人脸检测 (SCRFD500M) | 15 | 属性识别 (DeepMAR-ResNet50) |
| 3 | 旋转框检测 (YOLO26N-OBB) | 16 | PPU 流水线 (YOLOv7-PPU) |
| 4 | 姿态估计 (YOLOv8s-Pose) | 17 | 关键点检测 (SuperPoint) |
| 5 | 手部关键点 (HandLandmarkLite) | 18 | 物体姿态估计 (DOPE) |
| 6 | 人脸对齐 (3DDFA-V2-MobileNetV1) | 19 | 全景驾驶感知 (YOLOPv2) |
| 7 | 实例分割 (YOLOv8N-Seg) | 20 | 3D 目标检测 (SFA3D) |
| 8 | 语义分割 (DeepLabV3+MobileNet) | 21 | 手部检测 (MediaPipe Palm) |
| 9 | 分类 (ResNet50) | 22 | 图像检索 (CLIP RN50) |
| 10 | 深度估计 (YOLO26-Depth-S) | 23 | 视觉地点识别 (EigenPlaces R18) |
| 11 | 图像去噪 (DnCNN-50) | 24 | 行人重识别 (RepVGG-A0) |
| 12 | 超分辨率 (ESPCN-X4) | 25 | 图像抠图 (PP-Matting HRNet-W48) |

第二个菜单选择执行模式。模式 1 和 2 运行 C++ 可执行文件,模式 3 到 6 运行 3.4 节介绍的四种 Python 变体:

> ![](assets/sc-run_demo2.png)

第三个菜单选择输入:`1` 表示该演示程序的示例视频,`2` 表示其示例图像。

这三个答案也可以通过选项给出,从而跳过所有菜单:`--task`、`--mode` 和 `--input`。下一个单元格会打印启动器的帮助。当 `RUN_DEMO_LAUNCHER = True` 时,它接着会以模式 `2`(C++ 异步)在示例图像上启动演示程序 `0`(YOLOv7)。该演示程序会打开窗口,因此请只在桌面会话中设置此标志。

In [ ]:
RUN_DEMO_LAUNCHER = False   # True starts the demo below; it opens a window (desktop session only)
%cd {DX_APP_DIR}

!./run_demo.sh --help

if RUN_DEMO_LAUNCHER:
    # Demo 0 (Object Detection, YOLOv7), mode 2 (cpp_async), input 2 (image): no menus
    !./run_demo.sh --task 0 --mode 2 --input 2

### 2.6 使用 dx_tool.sh 查找、查看和导出示例

`scripts/dx_tool.sh` 用于管理示例。不带参数启动时,它会打开一个交互式菜单,但它的查询命令也可以直接运行,因此在笔记本中使用很方便:

| 命令 | 作用 |
|---|---|
| `dx_tool.sh list [FILTER]` | 列出示例,可按任务过滤(例如 `det`) |
| `dx_tool.sh search <KEYWORD>` | 按模型名称或任务查找示例 |
| `dx_tool.sh info <MODEL>` | 显示某个示例的文件、工厂类和 `config.json` |
| `dx_tool.sh run` / `dx_tool.sh bench` | 一次运行或基准测试多个示例(交互式过滤) |
| `dx_tool.sh extract` | 导出独立软件包(交互式) |

下一个单元格会搜索 YOLO26-S 的示例,并打印检测器的详细信息。`search` 会以 `<task>/<family>/<variant>` 的形式打印每个匹配项,这正是第 3 节中导出脚本所接受的形式。`info` 会显示默认的示例图像和视频,以及 `config.json` 中的分数阈值和 NMS 阈值等信息。

In [ ]:
%cd {DX_APP_DIR}

# Which examples exist for YOLO26-S? (C++ and Python, every task)
!./scripts/dx_tool.sh search yolo26-s

# Details of one variant: files, factory classes, and config.json
!./scripts/dx_tool.sh info yolo26-s_640x640

交互式菜单还提供导出功能。它会向您提问,因此请在终端(File > New > Terminal)中运行:

```bash
cd <DX_ALL_SUITE_DIR>/dx-runtime/dx_app
./scripts/dx_tool.sh
```

下一个单元格会用您的实际路径打印同样的命令。菜单出现后,请按其下方的步骤操作。

In [ ]:
# Interactive script: run these two commands in a terminal (File > New > Terminal).
print(f"cd {shlex.quote(str(DX_APP_DIR))}")
print("./scripts/dx_tool.sh")


菜单出现后:
- 选择 `2`(Extract Package)。
- 选择编程语言:`1` C++、`2` Python 或 `3` 两者。
- 以 `<task>/<family>/<variant>` 形式输入示例路径,例如 `object_detection/yolo26/yolo26-s_640x640`。按 Tab 键可补全路径。
- 按 Enter 接受默认输出目录 `output`。

> ![](assets/sc-extract.png)

截图来自较早的版本,当时路径只有两级(`object_detection/yolo26s`)。几秒钟后,软件包会被写入 `output/cpp/<task>/<family>/<variant>` 和 `output/py/<task>/<family>/<variant>`。

该菜单在后台调用 `scripts/extract_model_package.sh`。第 3 节直接使用该脚本,因为它通过参数接收模型和语言,因此可以在笔记本单元格中运行。

## 3. 运行提取出的独立示例

2.6 节展示了交互式导出。本节在笔记本中用 `scripts/extract_model_package.sh` 完成同样的事情,然后在 **DX-APP 目录树之外**构建并运行导出的软件包,就像在您自己的项目中那样:

| 小节 | 示例(`<task>/<family>/<variant>`) | 语言与变体 | 展示内容 |
|---|---|---|---|
| 3.2 | `object_detection/yolo26/yolo26-s_640x640` | Python,`*_sync.py` | 运行任意 Model Zoo 模型的最简单方式 |
| 3.3 | `object_detection/yolo26/yolo26-s_640x640` | C++,`*_sync` | 使用 CMake 构建导出的 C++ 软件包 |
| 3.4 | `pose_estimation/yolo26_pose/yolo26-s-pose_640x640` | Python,`*_async_cpp_postprocess.py` | 另一种任务,使用 C++ 后处理 |
| 3.5 | `object_detection/yolo26/yolo26-s_640x640` | Python,全部四种变体 | 通过测量确定使用哪种变体 |

每个导出的 Python 示例都会导入 `dx_engine` 和 `dx_postprocess`,它们由 DX-Runtime 安装到 `dx-runtime/venv-dx-runtime` 中。因此这些单元格会显式调用该解释器,并使用 `--no-prune` 导出 Python 软件包(3.1 节会解释原因)。在 DX-APP 目录树内部,如果不指定输入,示例会回退到默认的示例图像,但导出的软件包不包含示例图像。因此单元格会传入 `dx_app/sample/img` 中的一张图像,以无显示方式运行(`--no-display --save`),并在笔记本中显示保存的结果。

### 3.1 导出软件包

先从脚本自带的帮助开始。它列出了接下来的单元格使用的参数:以 `<task>/<family>/<variant>` 形式指定的示例(或以 `<task>/<family>` 指定某个模型系列的所有变体)、`--lang cpp|py|both`、`--output-dir` 和 `--no-prune`。

In [ ]:
!cd "{DX_APP_DIR}" && bash scripts/extract_model_package.sh --help


不加 `--output-dir` 时,脚本会在 `dx_app` 内部就地准备软件包;加上它后,软件包会被复制到 `<dir>/cpp/<task>/<family>/<variant>` 和 `<dir>/py/<task>/<family>/<variant>`。下一个单元格会下载两个 YOLO26 模型,并把三个软件包导出到本教程的 `workspace/` 目录中(该目录被 git 忽略):C++ 目标检测器,以及 Python 目标检测器和姿态估计器。

In [ ]:
STANDALONE_DIR = WORKSPACE_DIR / "standalone"
OUTPUT_DIR = WORKSPACE_DIR / "standalone-outputs"
DX_RUNTIME_PYTHON = DX_RUNTIME_DIR / "venv-dx-runtime" / "bin" / "python"   # has dx_engine and dx_postprocess
SAMPLE_IMG_DIR = DX_APP_DIR / "sample" / "img"
MODEL_DIR = DX_WORKSPACE_DIR / "res" / "models"

for path in (STANDALONE_DIR, OUTPUT_DIR):
    path.mkdir(parents=True, exist_ok=True)
if not DX_RUNTIME_PYTHON.is_file():
    raise FileNotFoundError(f"DX-Runtime Python environment not found: {DX_RUNTIME_PYTHON} (Tutorial 01, section 3)")

# Both models were downloaded in Tutorial 01 section 3.3; this only downloads what is missing.
!cd "{DX_APP_DIR}" && bash setup.sh --models yolo26-s_640x640 yolo26-s-pose_640x640 --no-force

# Export: object detection in C++ (pruned, the default) and in Python, pose estimation in Python.
# The Python packages use --no-prune; the next markdown cell explains why.
!cd "{DX_APP_DIR}" && bash scripts/extract_model_package.sh object_detection/yolo26/yolo26-s_640x640 --lang cpp --output-dir "{STANDALONE_DIR}"
!cd "{DX_APP_DIR}" && bash scripts/extract_model_package.sh object_detection/yolo26/yolo26-s_640x640 --lang py --output-dir "{STANDALONE_DIR}" --no-prune
!cd "{DX_APP_DIR}" && bash scripts/extract_model_package.sh pose_estimation/yolo26_pose/yolo26-s-pose_640x640 --lang py --output-dir "{STANDALONE_DIR}" --no-prune

!tree -L 5 "{STANDALONE_DIR}"

#### 裁剪后的软件包与 `--no-prune`

默认情况下,导出会**裁剪** `common/`:只复制该示例实际使用的共享文件,而不是整个运行时层。这样软件包更小、更易阅读,但只能运行这一个示例。`--no-prune` 会复制完整的 `common/`;如果您打算在同一个软件包中加入第二个模型,也需要使用它。上一个单元格中的 C++ 软件包是裁剪过的。下一个单元格会使用默认的裁剪方式再次导出 Python YOLO26-S 示例,并将其与完整副本进行比较。

> **DX-APP v3.3.0 注意事项:** 裁剪后的 **Python** 软件包缺少其 `config.json` 中指定的处理器类,因此会在启动时报错并停止:`VariantConfigError: processor class 'LetterboxPreprocessor' is exported by neither common.processors nor common.visualizers`。在此问题修复之前,请像本教程这样使用 `--no-prune` 导出 Python 软件包。裁剪后的 C++ 软件包不受影响。

In [ ]:
PRUNED_DIR = WORKSPACE_DIR / "standalone-pruned"
!cd "{DX_APP_DIR}" && bash scripts/extract_model_package.sh object_detection/yolo26/yolo26-s_640x640 --lang py --output-dir "{PRUNED_DIR}"


def count_files(folder):
    return sum(1 for p in Path(folder).rglob("*") if p.is_file())


PACKAGE = Path("py") / "object_detection" / "yolo26" / "yolo26-s_640x640"
print(f"\n{'package':<14}{'all files':>10}{'common/ files':>15}")
for label, root in (("pruned", PRUNED_DIR), ("--no-prune", STANDALONE_DIR)):
    folder = root / PACKAGE
    print(f"{label:<14}{count_files(folder):>10}{count_files(folder / 'common'):>15}")

#### 模型卡

每个示例文件夹都带有一张由 Model Zoo 生成的**模型卡**。`README.md` 描述该模型:它是什么、它的精度和速度及其数据来源、它的下载内容,以及如何运行它。Python 示例文件夹中还有 `model.html`(可在浏览器中离线查看的同一张模型卡)和 `result.webp`(参考结果)。导出的 Python 软件包会保留这三个文件。下一个单元格会显示您刚刚导出的软件包的参考图像和模型卡。

In [ ]:
import re

CARD_DIR = STANDALONE_DIR / "py" / "object_detection" / "yolo26" / "yolo26-s_640x640"
!ls "{CARD_DIR}"

display(Image(filename=str(CARD_DIR / "result.webp"), width=480))   # the reference result
card = (CARD_DIR / "README.md").read_text()
display(Markdown(re.sub(r"!\[[^\]]*\]\(result\.webp\)", "", card)))  # the card; its image is shown above
print("Offline HTML card:", CARD_DIR / "model.html")

### 3.2 示例 1:Python 独立运行,目标检测

`yolo26-s_640x640_sync.py` 是最基本的同步变体:纯 Python 后处理,一次处理一帧。它是最适合首先阅读的变体。DX-APP 中每个 Python 示例的参数都相同:`--model`、一个输入选项(`--image`、`--video`、`--camera`、`--rtsp`),以及用于无显示运行的 `--no-display --save`。

```bash
cd <workspace>/standalone/py/object_detection/yolo26/yolo26-s_640x640
<dx-runtime>/venv-dx-runtime/bin/python yolo26-s_640x640_sync.py \
    --model <workspace>/res/models/yolo26-s_640x640.dxnn \
    --image <dx_app>/sample/img/sample_street.jpg \
    --no-display --save --save-dir <output-dir>
```

In [ ]:
EXAMPLE_DIR = STANDALONE_DIR / "py" / "object_detection" / "yolo26" / "yolo26-s_640x640"
MODEL = MODEL_DIR / "yolo26-s_640x640.dxnn"
IMAGE = SAMPLE_IMG_DIR / "sample_street.jpg"

!cd "{EXAMPLE_DIR}" && "{DX_RUNTIME_PYTHON}" yolo26-s_640x640_sync.py --model "{MODEL}" --image "{IMAGE}" --no-display --save --save-dir "{OUTPUT_DIR}"

show_latest_image(OUTPUT_DIR)

### 3.3 示例 2:C++ 独立运行,目标检测

导出的 C++ 软件包包含 `yolo26-s_640x640_sync.cpp`、`yolo26-s_640x640_async.cpp`、共享的 `common/` 层,以及一个链接到已安装 DX-RT 的自动生成的 `CMakeLists.txt`。构建它就是标准的 CMake 流程;没有任何内容回指 DX-APP 目录树。

```bash
cd <workspace>/standalone/cpp/object_detection/yolo26/yolo26-s_640x640
mkdir -p build && cd build
cmake .. && make -j$(nproc)
./yolo26-s_640x640_sync -m <workspace>/res/models/yolo26-s_640x640.dxnn \
    -i <dx_app>/sample/img/sample_street.jpg --no-display --save --save-dir <output-dir>
```

请注意,C++ 示例的选项写作 `-m/--model_path` 和 `-i/--image_path`;`./yolo26-s_640x640_sync --help` 会列出它们。构建需要一到两分钟。

In [ ]:
CPP_DIR = STANDALONE_DIR / "cpp" / "object_detection" / "yolo26" / "yolo26-s_640x640"

!cd "{CPP_DIR}" && mkdir -p build && cd build && cmake .. && make -j$(nproc)

!cd "{CPP_DIR}/build" && ./yolo26-s_640x640_sync -m "{MODEL}" -i "{IMAGE}" --no-display --save --save-dir "{OUTPUT_DIR}"

show_latest_image(OUTPUT_DIR)

### 3.4 示例 3:Python 独立运行,姿态估计,异步 + C++ 后处理

每个 Python 示例都有四种变体。文件名会告诉您正在运行的是哪一种:

| 变体 | 后处理 | 线程模型 | 适用场景 |
|---|---|---|---|
| `*_sync.py` | 纯 Python | 同步 | 学习和调试逻辑 |
| `*_async.py` | 纯 Python | 异步 | 更高的吞吐量;部署时通常的起点 |
| `*_sync_cpp_postprocess.py` | C++ 绑定(`dx_postprocess`) | 同步 | 后处理较重,一次处理一帧 |
| `*_async_cpp_postprocess.py` | C++ 绑定(`dx_postprocess`) | 异步 | 后处理较重,且需要高帧率 |

> 在 DX-APP v3.2.x 中,`*_cpp_postprocess.py` 脚本实际上会悄悄地运行 Python 后处理器;v3.3.0 修复了这个问题。因此,用旧版本测得的结果在各变体之间不会显示出差异。

本示例切换到另一种任务,以及异步 C++ 后处理变体。命令形式不变:只有目录、脚本名和模型不同。

```bash
cd <workspace>/standalone/py/pose_estimation/yolo26_pose/yolo26-s-pose_640x640
<dx-runtime>/venv-dx-runtime/bin/python yolo26-s-pose_640x640_async_cpp_postprocess.py \
    --model <workspace>/res/models/yolo26-s-pose_640x640.dxnn \
    --image <dx_app>/sample/img/sample_people.jpg \
    --no-display --save --save-dir <output-dir>
```

In [ ]:
POSE_DIR = STANDALONE_DIR / "py" / "pose_estimation" / "yolo26_pose" / "yolo26-s-pose_640x640"
POSE_MODEL = MODEL_DIR / "yolo26-s-pose_640x640.dxnn"
POSE_IMAGE = SAMPLE_IMG_DIR / "sample_people.jpg"

!cd "{POSE_DIR}" && "{DX_RUNTIME_PYTHON}" yolo26-s-pose_640x640_async_cpp_postprocess.py --model "{POSE_MODEL}" --image "{POSE_IMAGE}" --no-display --save --save-dir "{OUTPUT_DIR}"

show_latest_image(OUTPUT_DIR)

### 3.5 先测量,再选择变体

四种变体中哪一种最快,取决于时间花在了哪里。设置环境变量 `DXAPP_PROGRESS=1` 后,每个示例大约每秒打印一行 `[PROGRESS]`,并在结束时打印最后一行。每一行都包含吞吐量(`fps`、`avg_fps`)以及每个阶段每帧的平均耗时:读取输入(`read_ms`)、预处理(`pre_ms`)、NPU 推理(`infer_ms`)和后处理(`post_ms`)。

下一个单元格会以无显示方式在同一段 588 帧的视频上运行全部四种 YOLO26-S 变体,并把每种变体的最后一行整理成表格。单次运行如下所示:

```bash
DXAPP_PROGRESS=1 <dx-runtime>/venv-dx-runtime/bin/python yolo26-s_640x640_sync.py \
    --model <workspace>/res/models/yolo26-s_640x640.dxnn --video <dx_app>/assets/videos/boat.mp4 --no-display
```

该单元格大约需要 40 秒。

In [ ]:
VIDEO = DX_APP_DIR / "assets" / "videos" / "boat.mp4"
VARIANT_SCRIPTS = [
    "yolo26-s_640x640_sync.py",
    "yolo26-s_640x640_async.py",
    "yolo26-s_640x640_sync_cpp_postprocess.py",
    "yolo26-s_640x640_async_cpp_postprocess.py",
]

rows = []
for script in VARIANT_SCRIPTS:
    log = !cd "{EXAMPLE_DIR}" && DXAPP_PROGRESS=1 "{DX_RUNTIME_PYTHON}" {script} --model "{MODEL}" --video "{VIDEO}" --no-display 2>&1
    progress = [line for line in log if line.startswith("[PROGRESS]")]
    if not progress:
        print("\n".join(log[-20:]))
        raise RuntimeError(f"{script} printed no [PROGRESS] line; see its output above")
    rows.append((script, dict(re.findall(r"(\w+)=([\d.]+)", progress[-1]))))

print(f"{'variant':<44}{'avg_fps':>9}{'pre_ms':>8}{'infer_ms':>10}{'post_ms':>9}")
for script, m in rows:
    print(f"{script:<44}{m['avg_fps']:>9}{m['pre_ms']:>8}{m['infer_ms']:>10}{m['post_ms']:>9}")

如何解读该表格(具体数值取决于主机 CPU 和 NPU 卡):

- **异步与同步对比:** 异步变体会让多帧同时在 NPU 上处理,因此能达到高得多的帧率。在一块 DX-M1 卡上,YOLO26-S 异步运行约为 200 FPS,同步运行约为 50 FPS。异步模式下 `infer_ms` 会变大,因为它测量的是每帧从提交到得到结果的周转时间,而不仅仅是 NPU 时间。
- **C++ 与 Python 后处理对比:** `cpp_postprocess` 变体可以缩短 `post_ms`,但 YOLO26-S 的后处理耗时远低于一毫秒,因此帧率几乎没有变化。当 `post_ms` 在每帧耗时中占比较大时(例如分割掩码、大量锚框或主机 CPU 较慢),C++ 后处理才会带来收益。
- **经验法则:** 从 `*_async.py` 开始,用 `DXAPP_PROGRESS=1` 进行测量,只有当后处理成为瓶颈时才切换到 `cpp_postprocess` 变体。

### 3.6 动手尝试:Model Zoo 中的任意模型

上述示例遵循同一种模式,并且该模式适用于全部 497 个变体:

1. 在 [Model Zoo](https://developer.deepx.ai/modelzoo/) 页面选择一个模型,并用 `./scripts/dx_tool.sh search <keyword>` 找到它的示例。
2. 就地运行它。在 DX-APP 目录树内部,模型和输入都是可选的:不指定 `--model` 时,示例会加载 `assets/models/<variant>.dxnn`(如果缺失则先下载);不指定输入时,它会使用其任务的默认示例图像。
3. 想在它的基础上开发时,导出它:`scripts/extract_model_package.sh <task>/<family>/<variant> --lang py --output-dir <dir>`。

下一个单元格会在 `config/model_registry.json` 中查找一个变体,并在不指定模型和输入的情况下就地运行它的同步 Python 示例,这是最简短的运行方式。它会显示结果并打印导出命令。把 `VARIANT` 改成其他值即可尝试另一个模型,例如 `resnet50_224x224`(分类)、`yolov12-n_640x640`(YOLOv12)或 `rtdetr-r18vd-6x_640x640`(RT-DETR)。尚未下载到本地的模型会先被下载,这需要几秒到一分钟。

In [ ]:
import json

VARIANT = "scrfd-500m_640x640"   # change me: any variant (the .dxnn file stem) from config/model_registry.json

registry = json.loads((DX_APP_DIR / "config" / "model_registry.json").read_text())
entry = next((e for e in registry if VARIANT in (e["variant"], e["model_name"])), None)
if entry is None:
    raise KeyError(f"{VARIANT} is not in the registry. Search with: cd {DX_APP_DIR} && ./scripts/dx_tool.sh search {VARIANT}")

task, family, variant = entry["task"], entry["family"], entry["variant"]
SCRIPT = f"src/python_example/{task}/{family}/{variant}/{variant}_sync.py"
print(f"{variant}: task={task}, family={family}, model file={entry['dxnn_file']}\n")

# The shortest run: no --model and no input, so the example uses its default model and sample image.
!cd "{DX_APP_DIR}" && "{DX_RUNTIME_PYTHON}" {SCRIPT} --no-display --save --save-dir "{OUTPUT_DIR}"
show_latest_image(OUTPUT_DIR)

print("\nExport it as a standalone package:")
print(f'cd "{DX_APP_DIR}" && bash scripts/extract_model_package.sh {task}/{family}/{variant} --lang py --output-dir "{STANDALONE_DIR}"')

## 4. 多模型图:无需编写代码即可串联模型

实际应用中经常需要对每一帧运行多个模型:检测器找到人,第二个模型读取他们的属性;或者手掌检测器找到手,关键点模型定位手指关节。DX-APP v3.3.0 为此新增了**多模型图**(multi-model graph)。您在 JSON 文件中描述流水线,由一个可执行文件来运行它:

- **节点**(nodes)是输入源和各个模型,每个模型以其变体名称指定;
- **边**(edges)将它们连接起来;带有 `"roi"` 的边会裁剪出第一个模型找到的每个框,并将裁剪结果送入下一个模型。

无需编辑 C++ 代码,也无需重新构建:要尝试其他模型,只需修改 JSON 文件中的一个字符串。可执行文件是 `bin/multi_model_graph_sync` 和 `bin/multi_model_graph_async`,示例图位于 `src/cpp_example/multi_model_graph/` 中。

| 选项 | 作用 |
|---|---|
| `--graph <file>` | 运行该图 |
| `--check <file>` | 在不使用 NPU 的情况下验证该图及其模型文件 |
| `--output <file>` | 保存合成后的结果图像 |
| `--report <file>` | 以 JSON 格式保存每个节点的结果 |
| `--list-models` | 列出图中可以使用的模型(`--consumes roi` 列出第二阶段模型) |

> Python 模块 `dx_graph` 可以在 Python 中运行同样的图。它只有在执行 `./build.sh --all` 时才会被构建,因此本教程使用可执行文件。

### 4.1 构建图可执行文件并下载模型

如果您的 DX-APP 构建中还没有这两个可执行文件,下一个单元格会构建它们,大约需要一分钟。然后它会下载本节的图所使用的三个额外模型:一个手掌检测器、一个手部关键点模型和一个行人属性模型。

In [ ]:
%cd {DX_APP_DIR}
GRAPH_DIR = DX_APP_DIR / "src" / "cpp_example" / "multi_model_graph"
GRAPH_OUTPUT_DIR = WORKSPACE_DIR / "graph-outputs"
GRAPH_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not (DX_APP_DIR / "bin" / "multi_model_graph_async").is_file():
    !./build.sh --target multi_model_graph_sync multi_model_graph_async
!ls bin/multi_model_graph_*

!bash setup.sh --models mediapipe-hand-detector_192x192 mediapipe-hands-lite_224x224 deepmar_resnet50_224x224 --no-force

### 4.2 两阶段级联:先手掌检测,再手部关键点

`hand_cascade.json` 是最小的示例图:

In [ ]:
!cat "{GRAPH_DIR}/hand_cascade.json"

源 `cam` 是一张示例图像。`palm` 在整帧中查找手。从 `palm` 到 `landmark` 的边带有 `"roi"`,因此每个手部框在向外扩展 15% 后被裁剪出来,并传给关键点模型。

`--check` 会在 NPU 上运行任何内容之前验证该图:它解析每个模型,显示其 `produces` 和 `consumes`,并确认 `.dxnn` 文件存在。随后 `--graph` 运行该图,`--output` 保存合成图像,`--report` 写出每个节点的结果。示例图使用相对于 `dx_app` 的路径,因此请在该目录下运行它们。

In [ ]:
%cd {DX_APP_DIR}

!./bin/multi_model_graph_async --check src/cpp_example/multi_model_graph/hand_cascade.json

!./bin/multi_model_graph_async --graph src/cpp_example/multi_model_graph/hand_cascade.json --output "{GRAPH_OUTPUT_DIR}/hand_cascade.jpg" --report "{GRAPH_OUTPUT_DIR}/hand_cascade.json"

display(Image(filename=str(GRAPH_OUTPUT_DIR / "hand_cascade.jpg"), width=480))

报告按帧列出每个节点的结果。在整帧上运行的模型位于 `nodes` 下;在裁剪图上运行的模型位于 `roi_nodes` 下,并附带每个裁剪图在帧中对应的框。下一个单元格会汇总第一帧的结果。

In [ ]:
report = json.loads((GRAPH_OUTPUT_DIR / "hand_cascade.json").read_text())
frame = report["frames"][0]
for name, node in frame["nodes"].items():
    payload = node["payload"]
    print(f"{name:<10} full frame  {payload['shape']:<10} items={len(payload.get('items', []))}")
for name, crops in frame.get("roi_nodes", {}).items():
    for crop in crops:
        box = [round(v) for v in crop["origin"]["src_box"]]
        print(f"{name:<10} crop        {crop['payload']['shape']:<10} from box {box}")

### 4.3 通过编辑 JSON 替换模型

`cascade_od_attr.json` 会检测人,并把每个人的裁剪图传给 `deepmar_resnet50_224x224`,这是一个行人属性模型(性别、年龄段、衣着等)。它的检测器是 `yolov8-n_640x640`,而本教程并未下载该模型。下一个单元格不去下载它,而是把该图的一个副本写入工作区,并把检测器替换为第 3 节中的 `yolo26-s_640x640`。由于该副本位于 `dx_app` 之外,单元格还会把源图像路径改为绝对路径。然后它会检查并运行编辑后的图;无需重新构建任何内容。

In [ ]:
graph = json.loads((GRAPH_DIR / "cascade_od_attr.json").read_text())
for node in graph["nodes"]:
    if node.get("type") == "source":
        node["uri"] = str(DX_APP_DIR / node["uri"])   # absolute path: the copy is outside dx_app
    if node["id"] == "od":
        node["model"] = "yolo26-s_640x640"           # the one-string model swap

MY_GRAPH = GRAPH_OUTPUT_DIR / "person_attributes_yolo26.json"
MY_GRAPH.write_text(json.dumps(graph, indent=2) + "\n")
!cat "{MY_GRAPH}"

!./bin/multi_model_graph_async --check "{MY_GRAPH}"
!./bin/multi_model_graph_async --graph "{MY_GRAPH}" --output "{GRAPH_OUTPUT_DIR}/person_attributes.jpg"

display(Image(filename=str(GRAPH_OUTPUT_DIR / "person_attributes.jpg"), width=640))

### 4.4 哪些模型适用,以及更多示例

并非每个模型都适用于每个位置。第二阶段模型必须能够接受裁剪图:它的 `consumes` 列必须是 `roi` 或 `either`。`--list-models --consumes roi` 会从注册表中筛选出这些模型;`file` 列显示 `.dxnn` 是否已在本地。下一个单元格会打印前几行和总数。

随后 `run_multi_model_demo.sh --list` 会列出示例图。它们涵盖级联(检测器为属性、重识别或分类模型提供输入)、扇出(在同一帧上进行检测、分割和姿态估计)、链式处理(在检测之前进行超分辨率或低光增强),以及用于过滤、计数、跟踪和检查区域或越线的逻辑节点。`bash run_multi_model_demo.sh --check` 会在不使用 NPU 的情况下验证所有示例图,并为每个缺失的模型打印下载命令。

In [ ]:
%cd {DX_APP_DIR}

# Second-stage candidates: the first lines, then the number of models
!./bin/multi_model_graph_async --list-models --consumes roi | head -12
!./bin/multi_model_graph_async --list-models --consumes roi | tail -n +3 | wc -l

# The sample graphs and the demo scenarios
!bash run_multi_model_demo.sh --list

# Validate every sample graph (no NPU use; long output):
#!bash run_multi_model_demo.sh --check

## 5. 故障排除

| 症状 | 看到的内容 | 可能原因 | 解决方法 |
|---|---|---|---|
| 找不到旧的可执行文件名 | `./bin/yolov7_async: No such file or directory` | DX-APP v3.3.0 以变体命名可执行文件 | 使用新名称,例如 `./bin/yolov7_640x640_async`;`ls bin` 会列出它们 |
| 缺少可执行文件 | `./bin/<variant>_sync: No such file or directory` | 默认构建(`--minimal`)只构建 26 个演示模型 | 在终端中于 `dx_app` 下运行 `./build.sh --target <variant>` 或 `./build.sh --all` |
| 模型在加载时被拒绝 | `.dxnn container v9 needs DX-RT >= 3.5.0, but this runtime is 3.4.1` 或 `Model file format version 9 is not supported` | Model Zoo `2_5_0` 的文件需要 DX-RT 3.5.0 或更高版本 | 更新 DX-Runtime(教程 01 第 3 节);`dxrt-cli --version` 会显示已安装的版本 |
| `import dx_engine` 失败 | `dx_engine and DX-RT versions do not match` | `venv-dx-runtime` 中的 `dx_engine` 是为另一个 DX-RT 版本构建的 | 重新安装 DX-Runtime,使两者都为 3.5.x |
| 模型无法加载 | `Failed to load model`,路径位于 `assets/models` 下 | 模型未下载 | 带上变体名称运行第 1 节的下载单元格,然后重新运行 |
| 图检查报告缺少模型 | `MISSING: 1 model file is not in .../assets/models` | 图中指定的模型未下载 | 运行 `--check` 打印出的下载命令,或运行 `bash setup.sh --models <variant> --no-force` |
| 没有输出窗口 | `cannot open display`,或单元格结束时没有出现窗口 | 没有图形会话(SSH 或无显示主机) | 在桌面会话中运行,或保留 `--no-display --save` 并查看保存的结果 |
| 无法打开摄像头 | `Could not open camera 0` | 没有 `/dev/video0`,或用户不在 `video` 组中 | `v4l2-ctl --list-devices`;`sudo usermod -aG video $USER` 然后重新登录 |
| `run_demo.sh` 在显示菜单前停止 | `Failed to download demo models.` | 自动下载演示模型时没有网络连接 | 连接网络,或在有网络时运行一次 `./setup.sh --demo-models` |
| 导出的 Python 示例在启动时停止 | `VariantConfigError: processor class '...' is exported by neither common.processors nor common.visualizers` | 软件包使用默认裁剪方式导出,缺少 `config.json` 中指定的类(DX-APP v3.3.0) | 使用 `--no-prune` 重新导出 |
| 导出的 Python 示例无法导入 `dx_engine` | `ModuleNotFoundError: No module named 'dx_engine'` | 使用了 Jupyter 的 Python 而不是 DX-Runtime 环境运行 | 像第 3 节的单元格那样使用 `<dx-runtime>/venv-dx-runtime/bin/python` |
| 导出的 C++ 软件包无法完成配置 | `Could not find a package configuration file provided by "dxrt"` | DX-RT 未安装,或安装在自定义前缀下 | 安装 DX-Runtime(教程 01 第 3 节),或在 `cmake` 命令行中添加 `-DCMAKE_PREFIX_PATH=<prefix>` |
| 导出的示例找不到输入 | `Input file not found: sample/img/...` | 软件包不包含示例图像 | 使用 `dx_app/sample/img` 中的文件通过 `--image`(Python)或 `-i`(C++)传入 |

## 6. 总结

### 6.1 已完成的 DX-APP 工作流程

```text
Load SDK paths
       │
       ▼
Download the selected models by variant name
       │
       ▼
Run C++ or Python examples headless and view the saved results
       │
       ├── Synchronous
       ├── Asynchronous
       └── PPU
       │
       ▼
Find an example with dx_tool.sh, export it, and measure its variants
       │
       ▼
Chain several models with a multi-model graph
```

### 6.2 学习成果一览

| 领域 | 您现在能做的事 |
|---|---|
| SDK 路径 | 从 `config.json` 加载 `DX_APP_DIR` 及相关路径 |
| 命名 | 根据 `<task>/<family>/<variant>` 中的变体名称找到示例、其可执行文件和脚本 |
| 模型准备 | 按变体名称只下载练习所需的模型和视频 |
| 执行模式 | 根据目标选择同步、异步或 PPU 执行 |
| 测量 | 在选择之前用 `DXAPP_PROGRESS=1` 比较四种 Python 变体 |
| 演示启动器 | 使用 `run_demo.sh` 启动演示程序,可通过菜单,也可通过 `--task`、`--mode` 和 `--input` |
| 独立代码 | 使用 `scripts/extract_model_package.sh` 导出任意模型的示例,阅读其模型卡,然后在 DX-APP 之外构建和运行 |
| Model Zoo 覆盖 | 通过 `config/model_registry.json` 或 `dx_tool.sh search` 找到任意 Model Zoo 模型对应的 DX-APP 示例 |
| 多模型图 | 检查、运行和编辑串联多个模型的图 JSON |

### 6.3 完成清单

- [ ] 确认 DX-RT 能够识别 DEEPX NPU,并且 DX-RT 为 3.5.0 或更高版本
- [ ] 准备所需的 DX-APP 模型和视频资源
- [ ] 至少运行一个端到端推理示例,并查看其保存的结果
- [ ] 比较可用的执行模式
- [ ] 使用 `dx_tool.sh search` 为目标任务找到一个参考实现
- [ ] 打开 `run_demo.sh`
- [ ] 在 DX-APP 目录树之外导出、构建并运行独立的 Python 和 C++ 示例
- [ ] 测量四种 Python 变体,并为您的模型选择一种
- [ ] 运行一个您自己选择的 Model Zoo 模型
- [ ] 运行一个多模型图,并替换其中的一个模型

> **下一步:** 继续学习教程 03,改造一个已训练的模型,将其编译为 DXNN,并集成到 DX-APP 运行时配置中。